In [1]:
from pyspark.sql import SparkSession

# original spark = SparkSession.builder.appName("SpeedDating").getOrCreate()

spark = (SparkSession.builder.appName("SpeedDating").config("spark.executor.memory", "4g").getOrCreate()) #intentando tener 4G de RAM

print("Spark version:", spark.version)
spark


Spark version: 3.5.0


In [2]:
import os

# Toma credenciales de entorno si existen; si no, usa las de docencia por defecto
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER", "dpladmin")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD", "dpladmin123")
MINIO_ENDPOINT   = os.getenv("MINIO_ENDPOINT", "http://minio:9000")

hconf = spark._jsc.hadoopConfiguration()
hconf.set("fs.s3a.access.key", MINIO_ACCESS_KEY)
hconf.set("fs.s3a.secret.key", MINIO_SECRET_KEY)
hconf.set("fs.s3a.endpoint", MINIO_ENDPOINT)
hconf.set("fs.s3a.path.style.access", "true")
hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hconf.set("fs.s3a.connection.ssl.enabled", "false")  # MinIO sin TLS en entorno local

print("S3A configurado contra:", MINIO_ENDPOINT)

S3A configurado contra: http://minio:9000


Cargamos las librerías y los datos

In [ ]:
RAW_PATH = "s3a://dpl/raw/speeddating.csv"

df_raw = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .option("sep", ",")
          .csv(RAW_PATH))

print("Registros:", df_raw.count())
df_raw.printSchema()
df_raw.show(5, truncate=False)


Leemos el CSV original con encabezados y detección automática de tipos.

In [ ]:
from pyspark.sql import functions as F

# Convertir género a binario
df = df_raw.withColumn(
    "gender",
    F.when(F.lower(F.trim(F.col("gender"))) == "male", 1)
     .when(F.lower(F.trim(F.col("gender"))) == "female", 0)
     .otherwise(None)
)

# Lista de representaciones comunes de nulos en texto
missing_tokens = ["", "NA", "NaN", "nan", "None", "NULL", "null", "?"]

for col_name in df.columns:
    df = df.withColumn(
        col_name,
        F.when(F.col(col_name).isin(missing_tokens), None).otherwise(F.col(col_name))
    )

total = df.count()
exprs = []
for c in df.columns:
    expr = (
        (F.count(F.when(F.col(c).isNull(), c)) / F.lit(total) * 100).alias(c)
    )
    exprs.append(expr)

na_percent_df = df.select(*exprs)
na_percent_pd = na_percent_df.toPandas().T
na_percent_pd.columns = ["porcentaje_NA"]
na_percent_pd = na_percent_pd.sort_values(by="porcentaje_NA", ascending=False)

print("Porcentaje de valores faltantes (tras normalizar nulos):")
print(na_percent_pd.head(25))

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType

for c, t in df.dtypes:
    if t == "string":
        df = df.withColumn(c, F.col(c).cast(FloatType()))

df.printSchema()

In [ ]:
if "expected_num_interested_in_me" in df.columns:
    df = df.drop("expected_num_interested_in_me")

In [ ]:
from pyspark.sql import functions as F
from functools import reduce

rangos = {
    "age": (18, 55),
    "wave": (1, 21),
    "gender": (0, 1),
    "age_o": (18, 55),
    "d_age": (0, 37),
    "samerace": (0, 1),
    "importance_same_race": (0, 10),
    "importance_same_religion": (0, 10),
    "pref_o_attractive": (0, 100),
    "pref_o_sincere": (0, 100),
    "pref_o_intelligence": (0, 100),
    "pref_o_funny": (0, 100),
    "pref_o_ambitious": (0, 100),
    "pref_o_shared_interests": (0, 100),
    "attractive_o": (0, 10),
    "sinsere_o": (0, 10),
    "intelligence_o": (0, 10),
    "funny_o": (0, 10),
    "ambitous_o": (0, 10),
    "shared_interests_o": (0, 10),
    "attractive_important": (0, 100),
    "sincere_important": (0, 100),
    "intellicence_important": (0, 100),
    "funny_important": (0, 100),
    "ambtition_important": (0, 100),
    "shared_interests_important": (0, 100),
    "attractive": (0, 10),
    "sincere": (0, 10),
    "intelligence": (0, 10),
    "funny": (0, 10),
    "ambition": (0, 10),
    "attractive_partner": (0, 10),
    "sincere_partner": (0, 10),
    "intelligence_partner": (0, 10),
    "funny_partner": (0, 10),
    "ambition_partner": (0, 10),
    "shared_interests_partner": (0, 10),
    "sports": (0, 10),
    "tvsports": (0, 10),
    "exercise": (0, 10),
    "dining": (0, 10),
    "museums": (0, 10),
    "art": (0, 10),
    "hiking": (0, 10),
    "gaming": (0, 10),
    "clubbing": (0, 10),
    "reading": (0, 10),
    "tv": (0, 10),
    "theater": (0, 10),
    "movies": (0, 10),
    "concerts": (0, 10),
    "music": (0, 10),
    "shopping": (0, 10),
    "yoga": (0, 10),
    "interests_correlate": (-1, 1),
    "expected_happy_with_sd_people": (0, 10),
    "expected_num_matches": (0, 20),
    "like": (0, 10),
    "guess_prob_liked": (0, 10),
    "met": (0, 1),
    "match": (0, 1)
}

total_inicial = df.count()
mask = None
fuera_rango = {}

for col, (minv, maxv) in rangos.items():
    if col in df.columns:
        cond = (F.col(col).isNull()) | ((F.col(col) >= minv) & (F.col(col) <= maxv))
        fuera_rango[col] = df.filter(~cond).count()
        if mask is None:
            mask = cond
        else:
            mask = mask & cond

df = df.filter(mask)

pref_cols = ["pref_o_attractive", "pref_o_sincere", "pref_o_intelligence", "pref_o_funny", "pref_o_ambitious", "pref_o_shared_interests"]
imp_cols = ["attractive_important", "sincere_important", "intellicence_important", "funny_important", "ambtition_important", "shared_interests_important"]

for cols in [pref_cols, imp_cols]:
    if all(c in df.columns for c in cols):
        df = df.withColumn("suma_tmp", sum(F.col(c) for c in cols))
        na_cond = reduce(lambda a, b: a | b, [F.col(c).isNull() for c in cols])
        df = df.filter((F.abs(F.col("suma_tmp") - 100) <= 1) | na_cond)
        df = df.drop("suma_tmp")

total_final = df.count()
print(f"Filas totales antes de limpiar: {total_inicial}")
print(f"Filas totales después de limpiar: {total_final}")
df_clean = df

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import pandas as pd
from pyspark.sql import functions as F

df_pd = df.toPandas()
num_cols = df_pd.select_dtypes(include=['number']).columns.tolist()
df_num = df_pd[num_cols]

imputer = IterativeImputer(max_iter=10, random_state=42)
df_imputed_array = imputer.fit_transform(df_num)

df_imputed = pd.DataFrame(df_imputed_array, columns=num_cols)
df_pd[num_cols] = df_imputed

df_clean = spark.createDataFrame(df_pd)

print("=== Valores faltantes DESPUÉS de imputar ===")
df_clean.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_clean.columns]).show()

In [ ]:
# Ruta adaptada a curated_imputed para mantener consistencia con el pipeline
CURATED_QS = "s3a://dpl/curated_imputed"

(df_clean
 .write.mode("overwrite")
 .format("parquet")
 .save(CURATED_QS)
)

print("Escritura OK en:", CURATED_QS)